#### Maintenant qu'on a analysé nos données on va pouvoir implémenter nos algorithmes de recommandation. On va utiliser plusieurs approches.

#### 1. Approche basée sur le contenu

#### La premiere approche serait de faire une recommandation basée sur l'audience d'un film. C'est a dire on recommande les films qui sont les plus appréciés des utilisateurs. Mais la il faudrait définir une métrique qui prenne en compte, et la note moyenne obtenue par le film, et le nombre de vote obtenu par le film. On utilise une moyenne de note pondérée inspirée de IMDb

$$ WR = \frac{v}{v+m}R+ \frac{m}{v+m}C $$

#### R: Note moyenne du film
#### v: Nombre de notes du film
#### m: Nombre minimal de notes choisi
#### C: Moyenne générale de toutes les notes

#### Plus le m est grand plus la note du film est rapproché de la note moyenne générale, moins le m est grand plus la note du film est proche de sa propre moyenne

In [1]:
## pour que cette fonction ne fasse pas trop de travail on va juste calculer movies stat et la note moyenne generale une seule fois dans le script 
## et on va utiliser ces valeurs dans notre fonction pour calculer le score
## Allure de mouvie_stats:
##      movieId     average_rating     rating_count
##          1           4.146846            2077
##          2           3.201141             701

def IMDb(movie_stats,C,k,m):

    """Cette fonction les identifiants des k films les plus pertinents selon le score d'IMDb de notre base de données"""
    ## movie stat est un dataframe qui regroupe par film(groupby) les données suivantes: la moyenne et le nombre de votes
    ## k represente les k_films avec le meilleur score qu'on veut retourner
    ## m agit comme un poids qui rapproche ou eloigne la moy des films de la moy generale
    
    ## on peut modifier le nom des colonnes si besoin
    v = movie_stats["rating_count"]
    R = movie_stats["average_rating"]

    scores = (v / (v + m)) * R + (m / (v + m)) * C

    return scores.nlargest(k).index.tolist()


In [28]:
## La on a une version beaucoup plus générale de la fonction 

def popularity_recommender(ratings, movies,m = 100, n=10):

    """
        Cette fonction les films les plus pertinents de notre base de données en fonction de la metrique de IMDb

        ratings: Le dataFrame qui represente les notations des films ["UserID", "MovieID", "Rating", "Timestamp"]
        movies: le dataFrame qui represente les films ["MovieID", "Title", "Type"]
        m: le facteur de pondération, selon que l'on veule que la note se rapproche plus de la moyenne globale ou de la moyenne du film lui même
        n: le nombre de films que l'on veut proposer
        
    """

    C = ratings["rating"].mean()
    

    stats = ratings.groupby("movieId").agg(
        average_rating=("rating", "mean"),
        rating_count=("rating", "count")
    ).reset_index()

    stats["score"] = (
        (stats["rating_count"] / (stats["rating_count"] + m))
        * stats["average_rating"]
        +
        (m / (stats["rating_count"] + m)) * C
    )

    result = stats[["movieId","score"]].merge(
        movies[["movieId", "title"]],
        on="movieId"
    )
    result = result.sort_values("score", ascending=False).head(n).reset_index(drop=True)
    return result

In [10]:
import pandas as pd

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["userID", "movieId", "rating", "timestamp"],
    encoding="latin-1"
)


movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "type"],
    encoding="latin-1"
)

In [29]:
recommandations = popularity_recommender(ratings,movies,n=20)
recommandations

,movieId,score,title
0,318,4.512744,"Shawshank Redemption, The (1994)"
1,858,4.484355,"Godfather, The (1972)"
2,527,4.471779,Schindler's List (1993)
3,50,4.467422,"Usual Suspects, The (1995)"
4,1198,4.443442,Raiders of the Lost Ark (1981)
5,2019,4.426039,Seven Samurai (The Magnificent Seven) (Shichin...
6,260,4.425479,Star Wars: Episode IV - A New Hope (1977)
7,1148,4.413601,"Wrong Trousers, The (1993)"
8,904,4.398397,Rear Window (1954)
9,745,4.396508,"Close Shave, A (1995)"
